In [ ]:
import gzip
import csv

with gzip.open("data/accessibilities/accessibilities_3utr.gz", "rt") as gzipFile:
    file = csv.reader(gzipFile, delimiter = "\t")

    first_line = None
    extremes = {}
    
    for line in file:
        if first_line == None:
            first_line = line
            continue

        if first_line[0] != line[0]:
            raise Exception("Different region names")

        if first_line[1] != line[1]:
            raise Exception("Different window size")

        if first_line[2] != line[2]:
            raise Exception("Different start")

        if first_line[3] != line[3]:
            raise Exception("Different end")

        if first_line[4] != line[4]:
            raise Exception("Different modifications")

        if first_line[5] != line[5]:
            raise Exception("Different footprint")

        if first_line[6] != line[6]:
            raise Exception("Different feature")

        if first_line[7] == line[7]:
            raise Exception("Same modification state")

        first_line_data = list(map(float, first_line[8].split(",")))
        line_data = list(map(float, line[8].split(",")))
        
        if len(first_line_data) != len(line_data):
            raise Exception("Different result size")

        region_name = line[0]
        window_size = line[1]
        start = line[2]
        end = line[3]

        if int(end) - int(start) > 5000:
            first_line = None
            continue
        
        modifications = list(map(int, line[4].split(",")))
        
        footprint = line[5]
        feature = line[6]
        
        mod_data = None
        unmod_data = None

        if first_line[7] == "mod":
            mod_data = first_line_data
            unmod_data = line_data
        else:
            mod_data = line_data
            unmod_data = first_line_data

        diff_data = [v for v in map(lambda pair: pair[1] - pair[0], zip(mod_data, unmod_data))]

        max_diff = max(diff_data)
        min_diff = min(diff_data)

        if (window_size, footprint, feature) in extremes:
            existing_extremes = extremes[(window_size, footprint, feature)]

            existing_max = existing_extremes["max"]
            existing_min = existing_extremes["min"]

            if max_diff > existing_max[0]:
                extremes[(window_size, footprint, feature)]["max"] = (max_diff, window_size, footprint, feature, region_name, start, end, modifications, unmod_data, mod_data, diff_data)

            if min_diff < existing_min[0]:
                extremes[(window_size, footprint, feature)]["min"] = (min_diff, window_size, footprint, feature, region_name, start, end, modifications, unmod_data, mod_data, diff_data)
        else:
            extremes[(window_size, footprint, feature)] = {}
            extremes[(window_size, footprint, feature)]["max"] = (max_diff, window_size, footprint, feature, region_name, start, end, modifications, unmod_data, mod_data, diff_data)
            extremes[(window_size, footprint, feature)]["min"] = (min_diff, window_size, footprint, feature, region_name, start, end, modifications, unmod_data, mod_data, diff_data)

        first_line = None
print("done")

In [ ]:
import gzip
import csv

with gzip.open("data/extremes/extremes_3utr.gz", "wt") as gzipFile:
    file = csv.writer(gzipFile, delimiter = "\t")

    for specification, min_max in extremes.items():
        for key, data in min_max.items():
            print(key, data[0], "window", data[1], "footprint", data[2], "feature", data[3], "region", data[4], "start", data[5], "end", data[6])

            modifications = data[7]
            unmod_data = data[8]
            mod_data = data[9]
            diff_data = data[10]

            modifications_str = ",".join([str(v) for v in modifications])
            unmod_str = ",".join([str(v) for v in unmod_data])
            mod_str = ",".join([str(v) for v in mod_data])
            diff_str = ",".join([str(v) for v in diff_data])
            
            file.writerow([key, data[0], data[1], data[2], data[3], data[4], data[5], data[6], modifications_str, unmod_str, mod_str, diff_str])

print("done")

In [ ]:
import gzip
import csv

global_min = None
global_max = None

def print_line(line):
    print(line[0], line[1], line[2], line[3], line[4], line[5], line[6], line[7], list(map(int, line[8].split(","))))

with gzip.open("data/extremes/extremes_3utr.gz", "rt") as gzipFile:
    file = csv.reader(gzipFile, delimiter = "\t")

    global_min = None
    global_max = None
    
    for line in file:
        if global_min == None or line[1] < global_min[1]:
            global_min = line

        if global_max == None or line[1] > global_max[1]:
            global_max = line

    print_line(global_min)
    print_line(global_max)
                

In [ ]:
import gzip
import csv
from fasta import FASTA
from src.files.files import get_files

get_files().get_assembled_region_fasta_files().get_3utr_file().open_or_recompute()

file = FASTA("data/regions_fasta/region_3utr.fna.gz")

for entry in file:
    region_name = entry.name[:entry.name.find(":")]

    if region_name == global_max[5]:
        max_seq = str(entry.seq).upper().replace("T", "U")
        max_start = int(global_max[6])
        max_end = int(global_max[7])
        max_seq_slice = max_seq[max_start:max_end]
        max_modifications = list(map(int, global_max[8].split(","))) 
        
        print("max", region_name)
        print(max_seq_slice)
        print()
        print(max_modifications)
        print()

        max_seq_modified = str(max_seq_slice)

        for max_modification in max_modifications:
            max_seq_modified = max_seq_modified[:max_modification] + "6" + max_seq_modified[max_modification + 1:]

        print(max_seq_modified)
        print()
    if region_name == global_min[5]:
        min_seq = str(entry.seq).upper().replace("T", "U")
        min_start = int(global_min[6])
        min_end = int(global_min[7])
        min_seq_slice = min_seq[min_start:min_end]
        min_modifications = list(map(int, global_min[8].split(","))) 
        
        print("min", region_name)
        print(min_seq_slice)
        print()
        print(min_modifications)
        print()
        
        min_seq_modified = str(min_seq_slice)

        for min_modification in min_modifications:
            min_seq_modified = min_seq_modified[:min_modification] + "6" + min_seq_modified[min_modification + 1:]

        print(min_seq_modified)
        print()
        

In [10]:
from src.files.files import get_files
from fasta import FASTA
import csv

get_files().get_assembled_region_fasta_files().get_exons_file().open_or_recompute()

file = FASTA("data/regions_fasta/region_exons.fna.gz")

region_sequences = {}

for entry in file:
    region_name = entry.name[:entry.name.find(":")]

    region_sequences[region_name] = str(entry.seq).upper().replace("T", "U")

for feature in ["external", "hairpin", "internal", "multibranch"]:
    for min_max in ["min", "max"]:
        data_file = open(f"data/access_{min_max}_{feature}.csv")
        csv_file = csv.reader(data_file, delimiter = "\t")

        min_length = None
        min_entry = None

        for csv_entry in csv_file:
            entry_length = int(csv_entry[4]) - int(csv_entry[3])
            
            if min_length == None or entry_length < min_length:
                min_length = entry_length
                min_entry = csv_entry

        min_region_name = min_entry[1]
        min_seq = region_sequences[min_region_name]

        min_modifications = map(int, min_entry[5].split(","))

        print(min_max, feature, min_region_name, min_entry[0], min_entry[2])
        print("")
        print(min_seq)

        min_seq_modified = min_seq
        
        for min_modification in min_modifications:
            min_seq_modified = min_seq_modified[:min_modification] + "6" + min_seq_modified[min_modification + 1:]

        print("")
        print(min_seq_modified)
        print("")
        


min external NM_199003.2_exon -0.3883203294229687 5

GAAGCUGCCUCCGCCAUCUUGGAGAUGGGAGACGGGCGAUGGCUGUGGUCCUUCUGCUAAUGCAAACAACAAAACGGGCACACUAGUCACCCCCGAGGGAGGCCACCAUCACUGUAACUGUUGGCCAAAGCUACAAAAGAAGCGAGGGAAUCCAACCGAGCGCAGCGACACUGAGAACAGCUUCCCCUGCCUUCUGCGGCGGCAGAAGUGAAGUGCCUGAGGACCGGAAGGAUGGUGCAGUCCUGCUCCGCCUACGGCUGCAAGAACCGCUACGACAAGGACAAGCCCGUUUCUUUCCACAAAAAGAAGAUCUUCUGGAGCCACAGGAACAGCUUCCCCCACCUCCUUUACCGCCUCCUGUUUCCCAGGUUGAUGCUGCUAUUGGAUUACUAAUGCCGCCUCUUCAGACCCCUGUUAAUCUCUCAGUUUUCUGUGACCACAACUAUACUGUGGAGGAUACAAUGCACCAGCGGAAAAGGAUUCAUCAGCUAGAACAGCAAGUUGAAAAACUCAGAAAGAAGCUCAAGACCGCACAGCAGCGAUGCAGAAGGCAAGAACGGCAGCUUGAAAAAUUAAAGGAGGUUGUUCACUUCCAGAAAGAGAAAGACGACGUAUCAGAAAGAGGUUAUGUGAUUCUACCAAAUGACUACUUUGAAAUAGUUGAAGUACCAGCAUAAAAAAAUGAAAUGUGUAUUGAUUUCUAAUGGGGCAAUACCACAUAUCCUCCUCUAGCCUGUAAAGGAGUUUCAUUUAAAAAAAUAACAUUUGAUUACUUAUAUAAAAACAGUUCAGAAUAUUUUUUUAAAAAAAAUUCUAUAUAUACUGUAAAAUUAUAAAUUUUUUUGUUUGUAAUUUCAGGUUUUUUACAUUUUAACAAAAUAUUUUAAAAGUUAUAAACUAACCUCAGACCUCUAAUGUAAGUUGGUUUCAAGAUUGGGGAUUUUGGG